In [1]:
import os
import itertools
import numpy as np
import pandas as pd

In [2]:
def read_chibio(file_name, emissions=None, overlay=False):
    m = pd.read_csv(file_name)
    
    if emissions is None:
        emissions = [x for x in m.columns
                     if x.startswith('FP')
                     and not x.endswith('base')]

    # get time in hours
    m['time'] = m['exp_time'] / 60 / 60
    # get log of the OD
    m['ln(od)'] = np.log(m['od_measured'])

    # create a timedelta index
    m.index = pd.to_timedelta(m['time'], unit='h')
    # smooth OD readings taking the median over a sliding 10-minute window
    m['smooth_od'] = m.rolling('60min', min_periods=5)['od_measured'].median()
    # smooth emission readings taking the median over a sliding 10-minute window
    for em in emissions:
        m[f'smooth_{em}'] = m.rolling('60min', min_periods=5)[em].median()
        m[f'smooth_norm_{em}'] = m[f'smooth_{em}'] / (m['smooth_od'] + 1)

    for em1, em2 in itertools.combinations(emissions, 2):
        m[f'{em1}_{em2}_ratio'] = m[em1] / m[em2]
        m[f'smooth_{em1}_{em2}_ratio'] = m[f'smooth_{em1}'] / m[f'smooth_{em2}']
        m[f'smooth_norm_{em1}_{em2}_ratio'] = m[f'smooth_{em1}_{em2}_ratio'] / (m['smooth_od'] + 1)
        m[f'smooth_{em1}_{em2}_proportion'] = m[f'smooth_{em1}'] / (m[f'smooth_{em2}'] + m[f'smooth_{em1}'])
        m[f'smooth_norm_{em1}_{em2}_proportion'] = m[f'smooth_{em1}_{em2}_proportion'] / (m['smooth_od'] + 1)
    for em2, em1 in itertools.combinations(emissions, 2):
        m[f'{em1}_{em2}_ratio'] = m[em1] / m[em2]
        m[f'smooth_{em1}_{em2}_ratio'] = m[f'smooth_{em1}'] / m[f'smooth_{em2}']
        m[f'smooth_norm_{em1}_{em2}_ratio'] = m[f'smooth_{em1}_{em2}_ratio'] / (m['smooth_od'] + 1)
        m[f'smooth_{em1}_{em2}_proportion'] = m[f'smooth_{em1}'] / (m[f'smooth_{em2}'] + m[f'smooth_{em1}'])
        m[f'smooth_norm_{em1}_{em2}_proportion'] = m[f'smooth_{em1}_{em2}_proportion'] / (m['smooth_od'] + 1)
    
    return m

In [3]:
def find_turbidostat(v):
    x = v[v['pump_1_rate'] > 0].sort_values('time')[['time',
                                                     'pump_1_rate']].reset_index(drop=True)
    x['delta'] = x['time'].diff()
    
    y = x[x['delta'] < 2].sort_values('time').reset_index(drop=True)
    y['delta2'] = y['time'].diff()

    if y.shape[0] == 0:
        return None
    
    times = []
    start = y['time'].values[0]
    end = None
    for time in y[y['delta2'] > 2]['time'].values:
        if start is None:
            start = time
            end = None
            continue
        end = time
        times.append((start, end))
        start = None
    if end is None:
        times.append((start, start + 2))

    return times

In [4]:
d = {
    'rep1': {
        'M1': 'TriComm KAN+PN23 2xMIC',
        'M2': 'TriComm KAN no drug',
        'M3': 'TriComm KAN 1xMIC',
        'M4': 'TriComm KAN 2xMIC',
        'M5': 'TriComm PN23 no drug', 
        'M6': 'TriComm PN23 1xMIC',
        'M7': 'TriComm PN23 2xMIC'
    },
    'rep2': {
        'M0': 'TriComm KAN no drug',
        'M1': 'TriComm PN23 no drug',
        'M2': 'TriComm KAN 1xMIC',
        'M3': 'TriComm PN23 1xMIC',
        'M4': 'TriComm KAN 2xMIC',
        'M5': 'TriComm PN23 2xMIC'
    }
}

In [5]:
EMISSIONS = ['FP1_emit1', 'FP2_emit1', 'FP3_emit1']

def parse_chibio(rep, chunks, time_add):
    last = {}
    res = []
    for folder in chunks:
        for f in os.listdir(f'{rep}/{folder}'):
            if f.endswith('_data.csv'):
                m = read_chibio(os.path.join(rep, folder, f),
                                emissions=['FP1_emit1',
                                           'FP2_emit1',
                                           'FP3_emit1'],
                                overlay=True)
                name = f.split('_')[3]
                if name not in d[rep]:
                    continue
                m['vial'] = name
                m['name'] = d[rep][name]
                m['short'] = d[rep][name].replace('TriComm ', '')
                m['plasmid'] = d[rep][name].split()[1]
                m['drug'] = 0 if 'MIC' not in d[rep][name] else int(d[rep][name].split('x')[0][-2:])
                m['replicate'] = rep
                m['time'] = m['time'] + last.get(name, 0) + time_add.get(folder, 0)
                last[name] = m['time'].values[-1]
                res.append(m.reset_index(drop=True))
    m = pd.concat(res).sort_values(['vial', 'time']).reset_index(drop=True)
    
    m['smooth_masked_FP1_emit1'] = m['smooth_FP1_emit1'].values
    m['smooth_masked_FP2_emit1'] = m['smooth_FP2_emit1'].values
    m['smooth_norm_masked_FP1_emit1'] = m['smooth_norm_FP1_emit1'].values
    m['smooth_norm_masked_FP2_emit1'] = m['smooth_norm_FP2_emit1'].values
    for em1, em2 in itertools.combinations(EMISSIONS, 2):
        m[f'masked_{em1}_{em2}_ratio'] = m[f'{em1}_{em2}_ratio']
        m[f'smooth_masked_{em1}_{em2}_ratio'] = m[f'smooth_{em1}_{em2}_ratio']
        m[f'smooth_norm_masked_{em1}_{em2}_ratio'] = m[f'smooth_norm_{em1}_{em2}_ratio']
    for vial in d:
        times = find_turbidostat(m[m['vial'] == vial])
        if times is None:
            continue
        for start, end in times:
            idx = m[(m['vial'] == vial) &
                    (m['time'] >= start - 0.5) &
                    (m['time'] <= end + 0.5)].index
            m.loc[idx, 'smooth_masked_FP1_emit1'] = np.nan
            m.loc[idx, 'smooth_masked_FP2_emit1'] = np.nan
            m.loc[idx, 'smooth_norm_masked_FP1_emit1'] = np.nan
            m.loc[idx, 'smooth_norm_masked_FP2_emit1'] = np.nan
            for em1, em2 in itertools.combinations(EMISSIONS, 2):
                m.loc[idx, f'masked_{em1}_{em2}_ratio'] = np.nan
                m.loc[idx, f'smooth_masked_{em1}_{em2}_ratio'] = np.nan
                m.loc[idx, f'smooth_norm_masked_{em1}_{em2}_ratio'] = np.nan
    
    cols = ['vial', 'replicate', 'name', 'short', 'plasmid', 'drug',
            'time', 'smooth_od', 'ln(od)', 'smooth_FP1_emit1', 'smooth_norm_FP1_emit1',
            'smooth_FP2_emit1', 'smooth_norm_FP2_emit1', 'smooth_FP3_emit1',
            'smooth_norm_FP3_emit1', 
            'smooth_masked_FP1_emit1', 'smooth_masked_FP2_emit1',
            'smooth_norm_masked_FP1_emit1', 'smooth_norm_masked_FP2_emit1',
            'FP1_emit1_FP2_emit1_ratio', 'smooth_FP1_emit1_FP2_emit1_ratio',
            'masked_FP1_emit1_FP2_emit1_ratio', 'smooth_masked_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_masked_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_FP1_emit1_FP2_emit1_proportion',
            'smooth_norm_FP2_emit1_FP1_emit1_proportion',
            'exp_time', 'od_measured', 'od_setpoint', 'od_zero_setpoint',
            'thermostat_setpoint', 'heating_rate', 'internal_air_temp',
            'external_air_temp', 'media_temp', 'opt_gen_act_int', 'pump_1_rate',
            'pump_2_rate', 'pump_3_rate', 'pump_4_rate', 'media_vol',
            'stirring_rate', 'LED_395nm_setpoint', 'LED_457nm_setpoint',
            'LED_500nm_setpoint', 'LED_523nm_setpoint', 'LED_595nm_setpoint',
            'LED_623nm_setpoint', 'LED_6500K_setpoint', 'laser_setpoint',
            'LED_UV_int', 'FP1_base', 'FP1_emit1', 'FP1_emit2', 'FP2_base',
            'FP2_emit1', 'FP2_emit2', 'FP3_base', 'FP3_emit1', 'FP3_emit2',
            'custom_prog_param1', 'custom_prog_param2', 'custom_prog_param3',
            'custom_prog_status', 'zigzag_target', 'growth_rate']
    m = m[cols].sort_values(['vial', 'time']).reset_index(drop=True)

    return m

m1 = parse_chibio('rep1', ['1st', '2nd'], {'2nd': 11})
# vial M4, replicate 1 (TriComm KAN 2xMIC) was confirmed contaminated: excluded
m1 = m1[m1['vial'] != 'M4'].reset_index(drop=True)
m2 = parse_chibio('rep2', ['1st'], {})

m = pd.concat([m1, m2]).reset_index(drop=True)

m.to_csv('chibio.tsv.gz', sep='\t', index=False)

/home/marco/.cache/uv/archive-v0/Wkjermz-TgwOSexvmOXAa/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
def parse_plate_reader(rep):
    pr1 = pd.read_excel(f'{rep}/Plate reader summary .xlsx',
                        sheet_name='523-620',
                        index_col=0)
    pr1.index.name = 'time'
    pr1 = pr1.stack().reset_index().rename(columns={'level_1': 'vial',
                                                    0: 'emission'})
    pr1['wavelength'] = '523-620'
    pr2 = pd.read_excel(f'{rep}/Plate reader summary .xlsx',
                        sheet_name='457-510',
                        index_col=0)
    pr2.index.name = 'time'
    pr2 = pr2.stack().reset_index().rename(columns={'level_1': 'vial',
                                                    0: 'emission'})
    pr2['wavelength'] = '457-510'
    pr = pd.concat([pr1, pr2])
    pr = pr[pr['vial'] != 'M9 media']
    
    pr['name'] = [d[rep][x] for x in pr['vial'].values]
    pr['short'] = [d[rep][x].replace('TriComm ', '') for x in pr['vial'].values]
    pr['time'] = [float(x.rstrip().replace('h', ''))
                  for x in pr['time'].values]
    pr['plasmid'] = [x.split()[1]
                     for x in pr['name'].values]
    pr['drug'] = [0 if 'MIC' not in x
                  else int(x.split('x')[0][-2:])
                  for x in pr['name'].values]
    
    pr['replicate'] = rep
    
    cols = ['vial', 'replicate', 'name', 'short', 'plasmid',
            'drug', 'time', 'emission', 'wavelength']
    pr = pr[cols].sort_values(['vial', 'time']).reset_index(drop=True)

    return pr

pr1 = parse_plate_reader('rep1')
# vial M4, replicate 1 (TriComm KAN 2xMIC) was confirmed contaminated: excluded
pr1 = pr1[pr1['vial'] != 'M4'].reset_index(drop=True)
pr2 = parse_plate_reader('rep2')

pr = pd.concat([pr1, pr2]).reset_index(drop=True)

pr.to_csv('plate_reader_fluorescence.tsv', sep='\t', index=False)

In [7]:
def parse_flow(rep):
    fl = pd.read_excel(f'{rep}/summary flowcyto with events.xlsx')
    
    fl['name'] = [d[rep][x] for x in fl['vial'].values]
    fl['short'] = [d[rep][x].replace('TriComm ', '') for x in fl['vial'].values]
    fl['time'] = [float(x.rstrip().replace('h', ''))
                 for x in fl['time'].values]
    fl['plasmid'] = [x.split()[1]
                    for x in fl['name'].values]
    fl['drug'] = [0 if 'MIC' not in x
                 else int(x.split('x')[0][-2:])
                 for x in fl['name'].values]
    
    fl = fl.set_index(['name', 'short', 'time', 'vial',
                     'plasmid', 'drug', 'Events']
                   ).stack().reset_index().rename(columns={'level_7': 'strain',
                                                           0: 'proportion',
                                                           'Events': 'total_events'})
    fl['proportion'] = [float(x.rstrip().replace('%',
                              '').replace(',',
                              '.').replace(' ', '')) / 100
                        for x in fl['proportion'].values]
    fl['events'] = (fl['proportion'] * fl['total_events']).astype(int)
    fl['log10(events)'] = np.log10(fl['events'] + 1)
    
    fl['replicate'] = rep
    
    cols = ['vial', 'replicate', 'name', 'short', 'plasmid',
            'drug', 'time',
            'total_events',
            'strain', 'events', 'log10(events)', 'proportion']
    
    fl = fl[cols].sort_values(['vial', 'time', 'strain']).reset_index(drop=True)

    return fl

fl1 = parse_flow('rep1')
# vial M4, replicate 1 (TriComm KAN 2xMIC) was confirmed contaminated: excluded
fl1 = fl1[fl1['vial'] != 'M4'].reset_index(drop=True)
fl2 = parse_flow('rep2')

fl = pd.concat([fl1, fl2]).reset_index(drop=True)

fl.to_csv('flow_cytometer.tsv', sep='\t', index=False)

In [8]:
def parse_counts(rep):
    c = pd.read_excel(f'{rep}/counts.xlsx')
    c = c.drop(columns=[x for x in c.columns if 'raw' in x])
    
    c['name'] = [d[rep][x] for x in c['vial'].values]
    c['short'] = [d[rep][x].replace('TriComm ', '') for x in c['vial'].values]
    c['time'] = [float(x.rstrip().replace('h', ''))
                 for x in c['time'].values]
    c['plasmid'] = [x.split()[1]
                    for x in c['name'].values]
    c['drug'] = [0 if 'MIC' not in x
                 else int(x.split('x')[0][-2:])
                 for x in c['name'].values]
    
    # Extract colony counts from Excel formulas
    import openpyxl
    import re
    wb = openpyxl.load_workbook(f'{rep}/counts.xlsx', data_only=False)
    ws = wb.active
    headers = [ws.cell(1, col).value for col in range(1, ws.max_column + 1)]
    isolate_cols = [h for h in headers if h not in ['vial', 'time', 'antibiotic']]
    
    count_rows = []
    for row_idx in range(2, ws.max_row + 1):
        row_vals = {headers[ci]: ws.cell(row_idx, ci + 1).value
                    for ci in range(len(headers))}
        if any(row_vals.get(k) is None for k in ['vial', 'time', 'antibiotic']):
            continue
        for isolate_col in isolate_cols:
            formula = row_vals.get(isolate_col)
            count_val = None
            if isinstance(formula, str) and formula.startswith('='):
                # Extract the numerator before the first /
                f = formula[1:]
                m = re.match(r'\(?(\d+(?:\.\d+)?)', f)
                count_val = float(m.group(1)) if m else None
            count_rows.append({
                'vial': row_vals['vial'],
                'time': row_vals['time'],
                'antibiotic': row_vals['antibiotic'],
                'isolate': isolate_col,
                'count': count_val,
            })
    count_long = pd.DataFrame(count_rows)
    count_long['time'] = [float(x.rstrip().replace('h', ''))
                          for x in count_long['time'].values]
    count_long['count'] = count_long['count'].fillna(0)
    
    c = c.set_index(['name', 'short', 'time', 'vial',
                     'plasmid', 'drug', 'antibiotic']
                   ).stack().reset_index().rename(columns={'level_7': 'isolate',
                                                           0: 'cfu_raw'})
    
    # Merge colony counts before renaming antibiotic -> plate
    c = c.merge(count_long, on=['vial', 'time', 'antibiotic', 'isolate'], how='left')
    c = c.rename(columns={'antibiotic': 'plate'})
    c['strain'] = [x[0] for x in c['isolate'].str.split('-')]
    
    c['replicate'] = rep
    
    cols = ['vial', 'replicate', 'name', 'short', 'plasmid',
            'drug', 'time', 'plate', 'strain',
            'count', 'cfu_raw']
    
    c = c[cols].sort_values(['vial', 'time', 'plate', 'strain']).reset_index(drop=True)

    return c

c1 = parse_counts('rep1')
# vial M4, replicate 1 (TriComm KAN 2xMIC) was confirmed contaminated: excluded
c1 = c1[c1['vial'] != 'M4'].reset_index(drop=True)
c = c1
c2 = parse_counts('rep2')

c = pd.concat([c1, c2]).reset_index(drop=True)

keys = ['replicate', 'vial', 'name', 'short', 'plasmid', 'drug', 'time', 'plate']

# cfu_raw / count recovers each strain's plate conversion factor (10^dilution / volume),
# i.e. the CFU/mL that a single colony of that strain represents. Strains may be scored
# at different dilutions, so raw counts cannot be summed across strains.
c['per_colony_cfu'] = np.where(c['count'] > 0, c['cfu_raw'] / c['count'], np.nan)

# detection limit = one colony on the LEAST diluted plate that was actually counted
# (the most sensitive plate read at this time point)
dlimit = c.groupby(keys)['per_colony_cfu'].min()
dlimit = dlimit.to_frame('detection_limit')
# default detection limit
dlimit = dlimit.fillna(100)

c = c.set_index(keys + ['strain']).join(dlimit).reset_index()

c['CFU/mL alt'] = [x if x >= y else y - (y / 2)
                   for x, y in c[['cfu_raw', 'detection_limit']].values]

c['CFU/mL'] = [x if x >= y else np.nan
               for x, y in c[['cfu_raw', 'detection_limit']].values]

c.to_csv('plate_counts.tsv', sep='\t', index=False)